# Report Reader
Run the cells in order with the `train.csv` and `test.csv` beside the notebook, or set `DATA_DIR` below. The final cell writes **`report_reader_outputs/submission.csv`**.


## Hospital reports
Extract inpatient revenue and annual admissions. Fiscal-year labels stay attached to their table rows. When a comparison gives two counts, the year and admissions context determine which count belongs to the reporting period. Monthly and daily rates are annualized.


In [ ]:
@dataclass(slots=True)
class Quantity:
    role: str
    value: float
    orig: bool = False
    revised: bool = False
    prior: bool = False
    launch: bool = False
    current: bool = False


# Parse tables, rank candidate quantities and recognize revisions
ORIG_CUE = re.compile(r"(?:first|originally|initially|earlier|previously|provisionally)\s+(?:been\s+)?"
                      r"(?:reported|stated|put|given|listed|recorded|announced|published|shown|estimated|filed|circulated|quoted|released|disclosed|noted|indicated|calculated|computed|tallied|logged|cited)|"
                      r"\b(?:original|initial|provisional)\s+(?:figure|number|count|estimate|version|value|report|reporting)|"
                      r"\bsuperseded\b(?!\s+by)|\bas\s+first\s+reported|\bdisregard|\bfirst\s+(?:\w+\s+){1,3}(?:was|were)\s+(?:put|reported|given|stated|recorded|estimated|placed)\b|\breported\s+(?:earlier|originally|initially|first)\b|\b(?:early|earlier|initial|first|previous|prior)\s+(?:reports?|estimates?|filings?|disclosures?)\b|"
                      r"\b(?:original|initial)\s+(?:\w+\s+){0,2}(?:price|rate|figure|number|count|estimate|value|total)\s+(?:of|was|at|stood)\b|"
                      r"\b(?:earlier|previous|prior)\s+(?:figure|number|estimate|value|count)\b(?=[^.;]{0,40}(?:reported|stated|given|was|had))", re.I)
REV_CUE = re.compile(r"revised|restated|corrected|replaced\s+by|superseded\s+by|now\s+stands|updated|amended|\brevision\b|\breconcil\w*|"
                     r"\breplaces\b|\bcorrect\s+(?:figure|count|number|value)|\btrue\s+(?:figure|count|number)", re.I)
NUM_RE = re.compile(r"(?<![\w.₹/-])(\d{1,3}(?:,\d{2,3})+|\d+)(?:\.(\d+))?(?![\d,]*\s*(?:crore|cr\b|lakh|million|billion|rupees|%))")

def pre_normalize(text):
    def _r(m):
        v = float(m.group(1).replace(',', '')) * 1000
        return f"{v:,.0f}" if abs(v - round(v)) < 1e-9 else f"{v:,.2f}"
    return re.sub(r"(?<![\w.])(\d[\d,]*(?:\.\d+)?)\s+thousand\b", _r, text, flags=re.I)

def num(m):
    """Convert a numeric match to float."""
    return float(m.group(0).replace(',', ''))

FY_LABEL = re.compile(r"\bFY\s*['’]?\d{2,4}(?:\s*[-–/]\s*\d{2,4})?\b|\b(?:19|20)\d{2}(?:\s*[-–/]\s*\d{2,4})?\b", re.I)

def is_val(c):
    c = re.sub(r"\bFY\s*['’]?\d{2,4}(?:\s*[-–/]\s*\d{2,4})?\b", " ", c, flags=re.I)
    if re.search(r'₹|\bRs\b|\bINR\b|rupees|crore|lakh|%|per\s+cent', c, re.I) and re.search(r'\d', c):
        return True
    c2 = re.sub(r'\b(?:FY\s*)?(?:19|20)\d\d(?:\s*-\s*\d\d)?\b', ' ', c)
    if not re.search(r'\d', c2):
        return False
    words = [w for w in re.findall(r'[A-Za-z]{3,}', c2)
             if w.lower() not in ('crore', 'lakh', 'million', 'billion', 'rupees', 'per', 'quintal', 'quintals', 'acre', 'acres',
                                  'tonnes', 'tonne', 'square', 'feet', 'foot', 'unit', 'apartment', 'apartments', 'inr')]
    return len(words) <= 3

def table_statements_all(rows):
    if not rows:
        return []
    hdr = rows[0]
    out = []
    h0 = hdr[0] if hdr and not GENERIC_HDR.match(hdr[0].strip()) else ''
    KEY = r"\s*(?:measure|metric|indicator|item|particulars?|parameter)\s*"
    VAL = r"\s*(?:value|figure|amount|count|number)\s*"
    allrows = [hdr] + rows[1:]
    if len(hdr) == 2 and any(re.fullmatch(KEY, r[0], re.I) for r in allrows) and any(re.fullmatch(VAL, r[0], re.I) for r in allrows):
        out2 = []
        for k in range(len(allrows) - 1):
            a0, b0 = allrows[k], allrows[k + 1]
            if len(a0) == 2 and len(b0) == 2 and re.fullmatch(KEY, a0[0], re.I) and re.fullmatch(VAL, b0[0], re.I):
                out2.append(f"{a0[1]}: {b0[1]}")
        others = [r for r in allrows if not (re.fullmatch(KEY, r[0], re.I) or re.fullmatch(VAL, r[0], re.I))]
        for r in others:
            if len(r) == 2 and is_val(r[1]) and not is_val(r[0]):
                out2.append(f"{r[0]}: {r[1]}")
        if out2:
            return out2
    for r in rows[1:]:
        if r and ((is_val(r[0]) and sum(is_val(x) for x in r) >= 2) or
                  (re.fullmatch(r"\s*(?:value|figure|amount|count|number|total)\s*", r[0], re.I) and len(rows) == 2 and hdr and
                   re.fullmatch(r"\s*(?:measure|metric|item|indicator|particulars?)\s*", hdr[0], re.I))):
            for j, cc in enumerate(r):
                if is_val(cc):
                    out.append(f"{hdr[j] if j < len(hdr) else ''}: {cc}".strip())
            continue
        labels = [h0] if h0 else []
        labels += [x for j2, x in enumerate(r) if j2 > 0 and not is_val(x)]
        if r and not is_val(r[0]):
            labels.append(r[0])
        only_first = is_val(r[0]) and not any(is_val(x) for x in r[1:])
        for j, c in enumerate(r):
            if (is_val(c) and j > 0) or (j == 0 and (len(r) == 1 or only_first)):
                h = hdr[j] if j < len(hdr) else ''
                hs = '' if GENERIC_HDR.match(h.strip()) else h
                lab = ' '.join(x for x in labels if x and not re.fullmatch(r'n/?a|-|—|not\s+shown|not\s+stated', x.strip(), re.I))
                out.append(f"{hs} {lab}: {c}".strip())
    return out

def report_units(text):
    return corporate_units(text, table_fn=table_statements_all)

def pick_quantity(qs, role, prefer_current=False, exclude_prior=False):
    """Select the preferred candidate value."""
    candidates = [
        q for q in qs
        if q.role == role
        and not q.orig
        and (not exclude_prior or not q.prior)
    ]
    if prefer_current:
        candidates = [q for q in candidates if not q.launch]
    if not candidates:
        return None
    revised = [q for q in candidates if q.revised]
    if revised:
        return revised[-1].value
    if prefer_current:
        current = [q for q in candidates if q.current]
        if current:
            return current[0].value
    return candidates[0].value

def orig_revised(pre, post=''):
    oc = [m.end() for m in ORIG_CUE.finditer(pre)
          if not (re.search(r"(?:replaces|replacing|supersedes|in\s+place\s+of)\s+(?:the\s+|that\s+|an?\s+)?\w*\s*$", pre[:m.start()], re.I)
                  and re.match(r"^\s*\w*\s*\)", pre[m.end():]))]
    rc = [m.end() for m in REV_CUE.finditer(pre)]
    orig = bool(oc) and (not rc or max(oc) > max(rc))
    if orig and re.search(r"\bcurrent\b|\bat\s+present\b|\bnow\b|\btoday\b|:\s", pre[max(oc):]):
        orig = False
    revised = (bool(rc) and (not oc or max(rc) > max(oc))) or bool(re.match(r"^[^.;]{0,40}\b(?:replaces|supersedes)\b", post, re.I))
    return orig, revised

REVISED_TO = re.compile(r"(?:revised|restated|corrected|updated|changed|amended|raised|lowered|cut)(?:\s+\w+){0,4}\s+(?:to|at)\s*$|"
                        r"\b(?:revision|correction|restatement)\s+(?:to|of)\s*$|"
                        r"(?:revised|corrected|restated|correct|true|actual|verified)\s+(?:\w+\s+)?(?:figure|number|count|total|value)\s+(?:is|of|was|stands\s+at|to\s+be|at|as)\s*$|"
                        r"(?:corrected|revised|restated)\s+(?:figure|number|count|value)?\s*$", re.I)

def clauses(s):
    parts = re.split(r";\s+|\s+(?=of\s+which\b)|,\s+(?=and\b)|\s+(?=while\b|whereas\b)|,\s+(?=(?:as\s+)?against\b|compared\s+(?:with|to)\b|versus\b|vs\.?\s|well\s+(?:above|below)\b|ahead\s+of\b|behind\b)", s)
    return [p for p in parts if p.strip()]

pick_current_period = partial(pick_quantity, exclude_prior=True)

# Hospital entities and year references
HOSP_RE = re.compile(r"\b((?:[A-Z][a-z]+\s+){1,2})(?:Multispeciality\s+|Multi-speciality\s+|Super\s+Speciality\s+|Speciality\s+|General\s+|Memorial\s+)?"
                     r"(?:Hospitals?|Healthcare|Medical\s+(?:Centre|Center|College)|Health|Institute|Medicity|Clinic|Care)\b")
HOSPITAL_OTHER_RE = re.compile(r"\bsister\b|\bpartner\s+(?:hospital|facility|institution)|\bneighbou?r(?:ing)?\b|\brival\b|\bnearby\s+hospital|"
                      r"\bother\s+hospital|\banother\s+hospital|\bpeer\s+hospital|\bcompetitor|\bshould\s+not\s+be\s+added|\botherhosp[a-z]\b|"
                      r"\bthat\s+hospital\b", re.I)
HOSPITAL_PRIOR_RE = re.compile(r"\b(?:previous|last|prior|preceding|earlier|past)\s+(?:financial\s+|fiscal\s+)?(?:year|period)|\ba\s+year\s+(?:earlier|ago|before)|"
                      r"\bthe\s+year\s+before\b|\byear\s+(?:immediately\s+)?(?:before|preceding)\b|\bcomparison\s+year\b|\bpreceding\s+year\b", re.I)
HOSPITAL_YEAR_RE = re.compile(r"(?<![\d.,₹])\b(?:FY\s?'?(\d{2})\b(?!\s*-)|FY\s?(20\d{2})(?!\s*-\s*\d)|(?:FY\s?)?(20\d{2})\s*-\s*(\d{2})\b|(?:fiscal|year\s+ended\s+March)\s+(20\d{2})|(20[12]\d)(?![.,]\d))", re.I)
EXCL_MONEY = re.compile(r"pharmacy|ancillary|grant|capital|construction|cardiac\s+wing|new\s+wing|operating\s+(?:expenses|costs|surplus)|expenditure|"
                        r"surplus|donation|research|net\s+profit|endowment|loan|debt|salary|salaries|equipment|cafeteria|diagnostic\s+(?:centre|income)|"
                        r"other\s+income|non-patient|investment", re.I)


def _years(s, shift=True):
    out = []
    for m in FY_SHORT_RANGE.finditer(s):
        out.append((m.start(), m.end(), 2000 + int(m[2])))
    for m in HOSPITAL_YEAR_RE.finditer(s):
        if any(a <= m.start() < b for a,b,_ in out):
            continue
        g = m.groups()
        if g[0]:
            y = 2000 + int(g[0])
        elif g[1]:
            y = int(g[1])
        elif g[2]:
            y = int(g[2]) + 1
        elif g[4]:
            y = int(g[4])
        else:
            pre = s[max(0, m.start() - 40):m.start()].lower()
            if re.search(r"(?:founded|established|set\s+up|opened|built|since|registered)\s+(?:in\s+)?$", pre):
                continue
            if re.match(r"^\s*(?:beds|patients|admissions|visits|staff|doctors|crore|lakh|rupees)", s[m.end():m.end() + 14], re.I):
                continue
            y = int(g[5])
        pre2 = s[max(0, m.start() - 30):m.start()]
        if shift and re.search(r"\b(?:before|preceding|prior\s+to|ahead\s+of|earlier\s+than)\s+(?:the\s+)?$", pre2, re.I):
            y -= 1
        elif shift and re.search(r"\b(?:after|following)\s+(?:the\s+)?$", pre2, re.I):
            y += 1
        if 2010 <= y <= 2035:
            out.append((m.start(), m.end(), y))
    return sorted(out)

# Extract inpatient revenue and annualize admissions
FY_SHORT_RANGE = re.compile(r"\bFY\s*['’]?(\d{2})\s*[-–/]\s*(\d{2})\b", re.I)

def year_near(s, start, end):
    """Bind a number to a year in its own clause. A directly attached year wins."""
    dates = _years(s, shift=False)
    ranked = []
    for left, right, year in dates:
        if left < end and right > start:
            continue
        gap = s[end:left] if left >= end else s[right:start]
        if re.search(r"[;.!?]|\b(?:and|but|while|whereas|versus|against|compared|to)\b|\d", gap, re.I):
            continue
        attached = left >= end and re.fullmatch(r"[ ,]*(?:in|for|during|of)?\s*", gap, re.I)
        ranked.append((0 if attached else 1, len(gap), year))
    return min(ranked)[2] if ranked else None


def admissions_comparison(s):
    """Resolve an annual count when a coordinated comparison omits the repeated subject."""
    if not re.search(r"admissions?|admitted|inpatients?", s, re.I) or HOSPITAL_OTHER_RE.search(s):
        return None
    if re.search(r"(?:per|a|each|every) (?:day|month|week)|/(?:day|month|week)|daily|monthly|weekly|originally|first reported", s, re.I):
        return None
    years = _years(s, shift=False)
    if len(set(y for _, _, y in years)) < 2 and not HOSPITAL_PRIOR_RE.search(s):
        return None
    # Preserve commas in thousands. Split on a comparison or a change of clause/subject.
    # Treat both sides of a "from ... to ..." count comparison as separate clauses.
    s = re.sub(r"(\bfrom\s+[\d,]+(?:\s+[^,;.]{1,35})?)\s+to\s+(?=\d)", r"\1 and moved to ", s, flags=re.I)
    parts = re.split(r"\s*(?:;|,?\s+\b(?:and|but|while|whereas|versus|compared (?:with|to)|as against|against)\b)\s*", s, flags=re.I)
    result = []
    for part in parts:
        named = re.search(r"(?:inpatient\s+)?admissions?|admitted|inpatients?", part, re.I)
        for m in NUM_RE.finditer(part):
            if any(a < m.end() and b > m.start() for a,b,_ in _years(part,False)):
                continue
            if any(a <= m.start() < b for a,b,*_ in find_all_money(part)):
                continue
            left = part[:m.start()]
            changed = re.search(r"\b(?:climbed|rose|fell|increased|moved|declined|decreased|grew|dropped|went|edged)\s+(?:(?:up|down)\s+)?to\s*$", left, re.I)
            if re.search(r"outpatient|\bbeds?\b|staff|surger|revenue|procedures?|occupancy", part, re.I):
                continue
            if not named and not changed and not result:
                continue
            year = year_near(part, m.start(), m.end())
            local_dates = {y for _,_,y in _years(part, False)}
            if year is None and len(local_dates) == 1:
                year = next(iter(local_dates))
            if year is None and HOSPITAL_PRIOR_RE.search(part):
                year = -1
            if year is not None and num(m) > 0:
                result.append((year, num(m)))
    if len({y for y,v in result}) > 1:
        latest = max(y for y,v in result)
        values = {v for y,v in result if y == latest}
        if len(values) == 1:
            return latest, values.pop()
    return None


def extract_hospital(text):
    text = normalize_text(pre_normalize(text))
    text = re.sub(r"\(?\s*(?:inpatient\s+(?:plus|and|\+)\s+outpatient|outpatient\s+(?:plus|and|\+)\s+inpatient)(?:\s+care|\s+revenue|\s+services)?\s*\)?", " combined ", text, flags=re.I)
    text = re.sub(r"combining\s+(?:in|out)patient\s+and\s+(?:in|out)patient(?:\s+care|\s+services)?", "combined", text, flags=re.I)
    names = []
    body = '\n'.join(ln for k, ln in enumerate(text.split('\n'))
                     if not re.match(r"^\s*(?:\*\*|#)", ln) and not (k == 0 and len(ln) < 160 and not ln.rstrip().endswith('.')))
    for m in HOSP_RE.finditer(body):
        words = m.group(1).split()
        key = words[-1]
        if key.lower() in ('the', 'our', 'a', 'this', 'each', 'every', 'its', 'which', 'that', 'neighbouring', 'nearby', 'other', 'another', 'per',
                           'total', 'inpatient', 'outpatient', 'patient', 'multispeciality', 'sister', 'partner', 'shri', 'sri', 'sanjeevani', 'city'):
            continue
        typ = 'OTHER' if words[0].lower() in ('sister', 'partner', 'neighbouring', 'rival') else 'NAME'
        if key not in [n for _, n in names]:
            names.append((typ, key))
    main = next((n for t, n in names if t == 'NAME'), None)
    others = [n for t, n in names if n != main]
    for k, w in enumerate(others):
        text = re.sub(rf"\b{re.escape(w)}\b", f'OtherHosp{chr(97 + k)}', text)
    us = report_units(text)
    all_years = [y for _, s, *_ in us for _, _, y in _years(s, shift=False)]
    cur_year = max(all_years) if all_years else None

    qs = []
    sec_other = False
    q_other = False
    last_other = False
    orig_role = None
    last_orig_money = None
    role_hint = None
    money_hint = None

    for kind, s, _, _, _ in us:
        if kind == 'heading':
            sec_other = bool(HOSPITAL_OTHER_RE.search(s)) and not (main and main in s)
            orig_role = None
            continue
        if kind == 'q':
            q_other = bool(HOSPITAL_OTHER_RE.search(s))
            last_other = q_other
            continue
        if kind != 'a':
            q_other = False
        sent_other = sec_other or (kind == 'a' and q_other)
        if kind != 'table' and last_other and re.match(r"^\s*(?:in|for|during|over)\s+[^,.;]{0,45}?,?\s+(?:it|its)\b", s, re.I):
            sent_other = True
        if kind != 'table' and last_other and re.match(r"^\s*(?:its|their|it|they|there|(?:in|for|during)\s+(?:fy\s?'?\d+|fiscal(?:\s+\d+)?|the\s+(?:same\s+)?year)(?:,)?\s+(?:it|its)\b|these\s+are)\b", s, re.I):
            sent_other = True
        if re.search(r"\bmain\s+hospital|\bour\s+(?:own\s+)?hospital|\bthe\s+main\s+(?:facility|institution)", s, re.I) and not sec_other:
            sent_other = False
        if kind == 'table' and not sec_other:
            sent_other = False
        if not sent_other:
            current = admissions_comparison(s)
            if current and (cur_year is None or current[0] == cur_year):
                qs.append(Quantity(role="adm_year", value=current[1], current=True))
        _syl = _years(s)
        _fv = min([mm[0] for mm in find_all_money(s)] + [mm.start() for mm in NUM_RE.finditer(s) if not (2005 <= num(mm) <= 2035)] + [len(s)])
        _sy = [y for p0, _, y in _syl]
        sent_year_prior = bool(_sy) and cur_year is not None and max(_sy) < cur_year and min(p0 for p0, _, _ in _syl) < _fv
        sent_prior = sent_year_prior or bool(re.match(r"^\s*(?:(?:in|during|for|over)\s+)?(?:the\s+)?(?:previous|prior|preceding|last|comparison)\s+(?:financial\s+|fiscal\s+)?year\b|"
                                   r"^[^.;]{0,15}?(?:in\s+|during\s+)?the\s+year\s+(?:immediately\s+)?(?:before|preceding)|"
                                   r"^\s*(?:a|one)\s+year\s+(?:earlier|before)|^\s*(?:by\s+way\s+of\s+|for\s+)comparison", s, re.I))
        for cl in clauses(s):
            other = sent_other
            if HOSPITAL_OTHER_RE.search(cl):
                other = True
                sent_other = True
            if other:
                continue
            money = find_all_money(cl)
            mspans = [(a, b) for a, b, _, _ in money]
            for k, (a, b, v, _) in enumerate(money):
                left = cl[(money[k - 1][1] if k > 0 else 0):a]
                right = cl[b:(money[k + 1][0] if k + 1 < len(money) else len(cl))]
                lab = left[-140:].lower()
                post = right[:80].lower()
                wide = (cl[max(0, a - 170):a]).lower()
                _contra = r"(?:not|never)\s+(?:\w+\s+){0,3}(?:within|in|of|part\s+of|as|into)\s+(?:the\s+)?(?:total\s+)?patient\s+revenue|" \
                          r"(?:different|separate|distinct)\s+(?:\w+\s+){0,3}from\s+(?:the\s+)?(?:total\s+)?patient\s+revenue|rather\s+than\s+patient\s+revenue|" \
                          r"(?:not|never)\s+(?:\w+\s+){0,2}patient\s+revenue|" \
                          r"(?:distinct|separate|different|apart)\s+(?:\w+\s+){0,2}from\s+(?:the\s+)?(?:in|out)patient\s+(?:revenue|services|care|stream)(?![^.;]*\b(?:which|that|whose|where)\b)|" \
                          r"(?:unrelated|unconnected|not\s+related)\s+(?:\w+\s+){0,2}to\s+(?:the\s+)?(?:\w+\s+){0,4}(?:in|out)?patient\s+revenue(?:\s+figures?)?"
                def _strip(t):
                    return re.sub(_contra, lambda mm: mm.group(0) if len(t) - mm.end() < 12 and re.match(r"^\s*(?:of|was|were|at|stood\s+at|:)?\s*$", t[mm.end():]) else " ", t)
                if re.search(r"\b(?:not|never)\s+(?:part of |included in )?(?:the )?(?:total )?patient revenue\s*[:,-]?\s*$",lab):
                    continue
                lab = _strip(lab)
                wide = _strip(wide)
                _pm0 = re.match(r"^\s*(?:was\s+|were\s+|is\s+)?(?:earned\s+|generated\s+|came\s+|derived\s+|contributed\s+|brought\s+in\s+)?(?:of\s+(?:revenue|income)\s+)?(?:from|in|of|for|by)\s+(?:the\s+)?(?:inpatient|outpatient|in-patient|out-patient)", post)
                _ex = max((mm.end() for mm in EXCL_MONEY.finditer(wide)), default=-1)
                _pos = max((mm.end() for mm in re.finditer(r"inpatient|outpatient|patient\s+revenue|combined", wide)), default=-1)
                if not _pm0 and (_ex > _pos or EXCL_MONEY.search(re.split(r"[;,]|\bwhile\b|\bwhereas\b|\band\b", post)[0][:40])):
                    continue
                orig, revised = orig_revised(cl[max(0, a - 120):a], right[:60])
                cues = [('inp_rev', r"inpatient|in-patient|admitted\s+patients|ward\s+revenue|from\s+inpatient"),
                        ('out_rev', r"outpatient|out-patient|opd\b|clinics?\s+revenue"),
                        ('tot_rev', r"total\s+patient\s+revenue|(?<!in)(?<!out)(?<!in-)(?<!out-)patient\s+revenue|total\s+revenue|(?<!in)(?<!out)patient\s+income|"
                                    r"\bcombined\b|overall\s+(?:patient\s+)?revenue")]
                pr = None
                pm = re.match(r"^\s*(?:was\s+|were\s+|is\s+)?(?:earned\s+|generated\s+|came\s+|derived\s+|contributed\s+|brought\s+in\s+)?(?:of\s+(?:revenue|income)\s+)?(?:from|in|of|for|by)\s+(?:the\s+)?(inpatient|outpatient|in-patient|out-patient)", post)
                if pm:
                    pr = 'inp_rev' if pm.group(1).startswith('in') else 'out_rev' if pm.group(1).startswith('out') else 'tot_rev'
                else:
                    best = None
                    for role, pat in cues:
                        ends = [mm.end() for mm in re.finditer(pat, wide)]
                        if role == 'inp_rev':
                            ends = [e for e in ends if not re.search(r"plus\s+outpatient", wide[e:e + 20])]
                        if ends and (best is None or max(ends) > best[0]):
                            best = (max(ends), role)
                    if best is not None:
                        pr = best[1]
                        if pr == 'inp_rev' and re.search(r"(?:share|%|per\s+cent)[^.]{0,40}inpatient", wide[-60:]) and re.search(r"(?<!in)patient\s+revenue", wide):
                            pr = 'tot_rev'
                _cont = False
                if pr is None and money_hint and re.match(r"^\s*(?:it|this|that|the\s+(?:figure|number|amount|total))\b", s, re.I):
                    pr = money_hint
                if pr is None and last_orig_money is not None and (REV_CUE.search((s[:s.find(cl)] if s.find(cl) >= 0 else '')[-80:] + lab[-80:]) or
                                                                   re.search(r"\b(?:replac\w*|supersed\w*)\b|final\s+figure", post[:60])):
                    pr, revised, _cont = last_orig_money, True, True
                if pr is None:
                    continue
                if not _cont and not re.search(r"revenue|income|earn|brought|generated|contributed|billing|receipts|came\s+to|totall?ed|stood|was|were|reached|rose|grew|"
                                 r"climbed|moved|amounted|services", wide[-120:] + ' ' + post[:40]):
                    continue
                seg_l = lab[-60:]
                seg_r = re.split(r"[;,]|\bto\b|\bwhile\b|\bbut\b", post)[0]
                prior = bool(HOSPITAL_PRIOR_RE.search(seg_r[:75])) or (bool(HOSPITAL_PRIOR_RE.search(seg_l)) and not re.search(r"\bto\s*$", seg_l))
                if re.search(r"\bfrom\s*$", lab[-8:]) and re.search(r"\bto\b", post[:80]) and not pm:
                    prior = True
                _is_to = bool(re.search(r"\bto\s*$", lab[-6:]))
                yrs_l = [y for p0, p1, y in _years(cl) if 0 <= a - p1 < 30 and not find_all_money(cl[p1:a])]
                yrs_r = [y for p0, p1, y in _years(cl) if 0 <= p0 - b < 45 and not find_all_money(cl[b:p0])]
                if HOSPITAL_PRIOR_RE.search(seg_l):
                    boundary = re.search(r",\s*(?:which|so|while|whereas)\b",right,re.I)
                    if boundary:
                        yrs_r = [y for p0,p1,y in _years(cl) if b <= p0 < b+min(45,boundary.start())]
                yy = yrs_r or yrs_l
                if yy and cur_year and max(yy) < cur_year and not (_is_to and not yrs_r):
                    prior = True
                if yy and cur_year and max(yy) == cur_year:
                    prior = False
                prior = prior or (sent_prior and not (yy and cur_year and max(yy) == cur_year))
                if _is_to and not HOSPITAL_PRIOR_RE.search(post[:50]) and not (yrs_r and cur_year and max(yrs_r) < cur_year):
                    prior = False
                if re.search(r"\b(?:previous|prior|preceding) (?:fiscal |financial )?year\s+(?:came|amounted) to\s*$",lab,re.I):
                    prior = True
                if orig:
                    last_orig_money = pr
                qs.append(Quantity(role=pr, value=v, orig=orig, revised=revised, prior=prior))

            for m in re.finditer(r"(\d+(?:\.\d+)?)\s*%", cl):
                if any(a0 <= m.start() < b0 for a0, b0 in mspans):
                    continue
                pre = cl[max(0, m.start() - 100):m.start()].lower()
                post = cl[m.end():m.end() + 90].lower()
                ctx = pre[-100:] + ' ' + post[:70]
                if re.search(r"occupanc|bed|growth|grew|rose|increase|fell|decline|margin|icu|mortality|utilisation|utilization", pre[-40:] + post[:25]):
                    continue
                if not re.search(r"revenue|income|billing|\btotal\b|\bshare\b|contribut|accounted|of\s+which", ctx):
                    continue
                ip = [mm for mm in re.finditer(r"inpatient|in-patient", ctx)]
                op = [mm for mm in re.finditer(r"outpatient|out-patient", ctx)]
                c0 = len(pre[-100:])
                ib = [c0 - mm.start() for mm in ip if mm.start() < c0]
                ob = [c0 - mm.start() for mm in op if mm.start() < c0]
                if ib or ob:
                    role = 'inp_share' if (min(ib) if ib else 999) <= (min(ob) if ob else 999) else 'out_share'
                else:
                    di = min((abs(mm.start() - c0) for mm in ip), default=999)
                    do = min((abs(mm.start() - c0) for mm in op), default=999)
                    if di == 999 and do == 999:
                        continue
                    role = 'inp_share' if di <= do else 'out_share'
                prior = bool(HOSPITAL_PRIOR_RE.search(pre[-50:])) or sent_prior
                qs.append(Quantity(role=role, value=float(m.group(1)) / 100, prior=prior, orig=False, revised=False))

            for m in NUM_RE.finditer(cl):
                if any(a < m.end() and b > m.start() for a,b,_ in _years(cl,False)):
                    continue
                if any(a0 <= m.start() < b0 for a0, b0 in mspans):
                    continue
                v = num(m)
                if 2005 <= v <= 2035 and ',' not in m.group(0):
                    continue
                pre = cl[max(0, m.start() - 110):m.start()].lower()
                post = cl[m.end():m.end() + 80].lower()
                if re.match(r"^\s*%", post):
                    continue
                if re.match(r"^\s*(?:beds?|doctors?|nurses?|staff|surgeons?|surger|surgical|procedures?|operations?|outpatient|visits?|consultations?|icu|cities|departments?|"
                            r"specialit|years?|crore|lakh|million|billion|km|acres?)\b", post):
                    continue
                after = re.split(r"[;]|\bwhile\b|\bwhereas\b", post)[0]
                lab = re.sub(r"\s+", " ", pre)[-110:]
                if re.match(r"^\s*(?:and|but|while|whereas)\b", cl, re.I) and s.find(cl) > 0:
                    lab = re.sub(r"\s+", " ", s[:s.find(cl)] + cl[:m.start()])[-110:].lower()
                lab_ny = HOSPITAL_YEAR_RE.sub(" ", lab)
                is_adm = bool(re.match(r"^\s*(?:inpatient\s+|in-patient\s+|patient\s+|new\s+)?(?:admissions?|inpatients?|patients\s+(?:were\s+)?admitted|in-patients)\b", post)) or \
                    bool(re.search(r"(?:admission|admitted|inpatients?\s+(?:treated|admitted)|treated\s*$)[^.;]{0,105}$", lab_ny)) and \
                    not re.search(r"(?:outpatient|visit|consultation|surger|surgic|caseload|procedure|bed|doctor|staff)[^.;]{0,25}$", lab[-30:])
                if not is_adm and role_hint and re.match(r"^\s*(?:it|this|that|the\s+(?:figure|number|count|total))\b", s, re.I) and \
                        not re.search(r"revenue|crore|lakh|billion|million", s[:m.start() + 1], re.I):
                    is_adm = True
                    _forced = role_hint
                else:
                    _forced = None
                if not is_adm:
                    if orig_role is not None and orig_role.startswith('adm') and (REVISED_TO.search(pre[-60:]) or
                                                                                  re.match(r"^\s*(?:\w+\s+){0,2}(?:replaces|supersedes)\b", post)):
                        role = orig_role
                    else:
                        continue
                elif _forced:
                    role = _forced
                else:
                    ctxp = lab_ny[-110:] + ' ' + after[:60]
                    role = 'adm_month' if re.search(r"per\s+month|a\s+month|each\s+month|every\s+month|monthly|/\s*month|month\s+on\s+average", ctxp) else \
                        'adm_day' if re.search(r"per\s+day|a\s+day|each\s+day|every\s+day|daily|/\s*day", ctxp) else 'adm_year'
                orig, revised = orig_revised(pre[-120:], post[:60])
                if orig:
                    orig_role = role
                elif orig_role == role and REV_CUE.search(pre[-70:]):
                    revised = True
                seg_r = re.split(r"[;,]|\bto\b|\bwhile\b|\bbut\b", post)[0]
                prior = bool(HOSPITAL_PRIOR_RE.search(seg_r[:70])) or (bool(HOSPITAL_PRIOR_RE.search(lab[-60:])) and not re.search(r"\bto\s*$", lab))
                if re.match(r"^\s*(?:compared\s+(?:with|to)|as\s+against|against|versus|up\s+from|down\s+from)\b", cl, re.I):
                    prior = True
                if re.search(r"\bfrom\s*$", lab[-8:]) and re.search(r"\bto\b", post[:60]):
                    prior = True
                _is_to_c = bool(re.search(r"\bto\s*$", pre[-6:]))
                yrs_l = [y for p0, p1, y in _years(cl) if 0 <= m.start() - p1 < 40 and not _is_to_c]
                yrs_r = [y for p0, p1, y in _years(cl) if 0 <= p0 - m.end() < 40 and not [mm for mm in NUM_RE.finditer(cl[m.end():p0]) if not (2005 <= num(mm) <= 2035)]]
                if HOSPITAL_PRIOR_RE.search(pre) and re.match(r"^\s*(?:inpatient\s+)?admissions?\s*,\s*(?:so|which|although|whereas|while)\b",post,re.I):
                    # A new clause may mention the reporting year after a prior-year count.
                    boundary = re.search(r",\s*(?:so|which|although|whereas|while)\b",post,re.I).start()
                    yrs_r = [year for a,b,year in _years(cl) if m.end() <= a < m.end()+boundary]
                yy = yrs_r or yrs_l
                if yy and cur_year and max(yy) < cur_year:
                    prior = True
                if yy and cur_year and max(yy) == cur_year:
                    prior = False
                prior = prior or (sent_prior and not (yy and cur_year and max(yy) == cur_year))
                qs.append(Quantity(role=role, value=v, orig=orig, revised=revised, prior=prior))

        last_other = sent_other
        if not re.search(r"\d", HOSPITAL_YEAR_RE.sub(" ", s)):
            money_hint = ('inp_rev' if re.search(r"inpatient\s+(?:services\s+)?revenue|revenue\s+from\s+inpatient|inpatient\s+services?\s+(?:earned|brought|generated)", s, re.I) else
                          'out_rev' if re.search(r"outpatient\s+(?:services\s+)?revenue|revenue\s+from\s+outpatient", s, re.I) else
                          'tot_rev' if re.search(r"total\s+patient\s+revenue|(?<!in)(?<!out)\bpatient\s+revenue", s, re.I) else None)
        else:
            money_hint = None
        if not re.search(r"\d", HOSPITAL_YEAR_RE.sub(" ", s)) and re.search(r"admission|admitted", s, re.I):
            role_hint = 'adm_month' if re.search(r"per\s+month|monthly", s, re.I) else 'adm_day' if re.search(r"per\s+day|daily", s, re.I) else 'adm_year'
        else:
            role_hint = None
    return qs

def parse_hospital(text):
    qs = extract_hospital(text)
    inp = pick_current_period(qs, 'inp_rev')
    if inp is None:
        tot = pick_current_period(qs, 'tot_rev')
        if tot is not None:
            sh, osh, out = pick_current_period(qs, 'inp_share'), pick_current_period(qs, 'out_share'), pick_current_period(qs, 'out_rev')
            if sh is not None:
                inp = tot * sh
            elif out is not None:
                inp = tot - out
            elif osh is not None:
                inp = tot * (1 - osh)
    dated = [q.value for q in qs if q.role == 'adm_year' and q.current and not q.orig and not q.prior]
    adm = dated[-1] if dated and not any(q.revised and q.role == 'adm_year' and not q.prior and not q.orig for q in qs) else pick_current_period(qs, 'adm_year')
    if adm is None:
        mth = pick_current_period(qs, 'adm_month')
        if mth is not None:
            adm = mth * 12
        else:
            day = pick_current_period(qs, 'adm_day')
            if day is not None:
                adm = day * 365
    x = np.array([np.nan if inp is None else inp, np.nan if adm is None else adm], float)
    return x

HOSPITAL_LABELS = {
    "revenue": r"inpatient (?:services?|care)?\s*(?:revenue|income)|revenue from inpatient|inpatient services|inpatient care|brought in|earned",
    "total": r"total patient revenue|patient revenue total",
    "outpatient": r"outpatient (?:services?)?\s*(?:revenue|income)|revenue from outpatient|outpatient services",
    "admissions": r"inpatient admissions|admissions|admitted",
    "share": r"inpatient|admitted patient",
}

def recover_hospital(text, inputs):
    """Fill missing fields using label distance, table position and correction cues."""
    scored = defaultdict(list)
    for line in normalize_text(pre_normalize(text)).splitlines():
        if re.search(r"sister|neighbour|rival|partner hospital", line, re.I):
            continue
        table = line.strip().startswith("|")
        sentences = [" | ".join(v.strip() for v in line.strip().strip("|").split("|"))] if table else split_sentences(line)
        for sentence in sentences:
            lower = sentence.lower()
            comparison = admissions_comparison(sentence)
            if comparison:
                scored["admissions"].append((comparison[0], float("inf"), comparison[1]))
            money = find_all_money(sentence)
            quantities = [(a,b,v,True) for a,b,v,_ in money]
            for m in re.finditer(r"\d[\d,]*(?:\.\d+)?", sentence):
                if not any(a <= m.start() < b for a,b,_,_ in money):
                    quantities.append((m.start(), m.end(), float(m[0].replace(",","")), False))
            for a,b,v,is_money in quantities:
                local = lower[max(0,a-82):a+85]
                context = lower[max(0,a-65):a] + " " + lower[b:b+65]
                adjust = 10 * table - 9 * bool(re.search(r"prior|previous|last year|year before|year earlier|initial|old figure",context))
                adjust += 4 * bool(re.search(r"revised|restated|corrected|updated|replaces|final|current",context))
                adjust -= 10 * bool(re.search(r"first reported|original|initially",context))
                if table and " | " not in sentence[:a]:
                    adjust -= 8
                for role, pattern in HOSPITAL_LABELS.items():
                    if is_money != (role in ("revenue","total","outpatient")):
                        continue
                    matches = list(re.finditer(pattern,lower))
                    if not matches:
                        continue
                    dist = min(max(m.start()-b, a-m.end(), 0) for m in matches)
                    score = adjust - min(dist,180)*.15
                    val = v
                    if role == "revenue":
                        if "inpatient" not in lower: continue
                        score -= 24 * ("outpatient" in local and "inpatient" not in local)
                        score -= 15 * bool(re.search("patient revenue|total revenue",local))
                    if role == "share":
                        if "%" not in lower or not 10 <= v <= 99: continue
                        val /= 100
                    if role == "admissions":
                        score -= 30 * (v < 400 or v > 250000)
                        score -= 45 * any(a0 <= a < b0 for a0,b0,_ in _years(sentence,shift=False))
                        score -= 16 * ("outpatient" in local)
                        score -= 10 * bool(re.search("previous|prior|year earlier|last year|year before",local))
                        for frequency, periods in ((r"per day|daily|each day",365), (r"per month|monthly|each month",12), (r"per week|weekly|each week",52)):
                            if re.search(frequency,lower):
                                val *= periods
                                score += 3
                                break
                        if not 400 <= val <= 200000: continue
                    if role == "admissions" and any(l < b and r > a for l,r,_ in _years(sentence,False)):
                        continue
                    year = year_near(sentence, a, b)
                    scored[role].append((year or 0, score, val))

    def best(role):
        return max(scored[role], key=lambda item:item[:2])[2] if scored[role] else np.nan
    revenue = best("revenue")
    if not np.isfinite(revenue):
        total, share = best("total"),best("share")
        revenue = total*share if np.isfinite(share) else total-best("outpatient")
    return np.where(np.isfinite(inputs),inputs,[revenue,best("admissions")])


## Fit each domain formula
Use all labeled reports. Hospital uses a log-linear power law. The other domains use monetary feature combinations and relative residuals with a scale floor near zero. Hospital and Real Estate use a three-robust-scale trimmed fit, Farmer uses Tukey biweight, and Corporate uses Cauchy weights. Iteration stops at a relative coefficient change of $10^{-9}$ or 50 updates.


In [ ]:
def formula_inputs(x, domain):
    if domain == "Hospital":
        return np.column_stack([np.ones(len(x)), np.log(x[:, 0] / 1e9), -np.log(x[:, 1] / 5e4)])
    if domain == "Farmer":
        return np.column_stack([x[:, 0] * x[:, 1], -x[:, 2]]) / 1000
    if domain == "Real Estate":
        return np.column_stack([x[:, 0] * x[:, 2], -x[:, 1] * x[:, 2]]) / 1e7
    return x  # Corporate operating profits are already in crore.



def fit_formula(x, target, domain):
    design = formula_inputs(x, domain)
    hospital = domain == "Hospital"
    y = np.log(target) if hospital else target
    bounds = ([-np.inf, .01, .01], [np.inf, 3, 3]) if hospital else (1e-9, np.inf)
    scale = np.ones_like(y) if hospital else np.maximum(abs(y), .5)

    def solve(weights):
        return lsq_linear(design * weights[:, None], y * weights, bounds=bounds).x

    theta = solve(1 / scale)
    for _ in range(50):
        fitted = design @ theta
        if not hospital:
            scale = np.maximum(abs(fitted), .5)
        residual = (fitted - y) / scale
        median = np.median(residual)
        spread = max(1e-4, 1.4826 * np.median(abs(residual - median)))
        if domain == "Corporate":
            weights = 1 / (1 + (residual / (2.385 * spread))**2)
        elif domain == "Farmer":
            weights = np.maximum(0, 1 - (residual / (4.685 * spread))**2)**2
        else:
            weights = abs(residual - median) <= 3 * spread
        update = solve(np.sqrt(weights) / scale)
        if np.max(abs(update - theta) / np.maximum(1, abs(theta))) <= 1e-9:
            return update
        theta = update
    raise RuntimeError(f"{domain} fit did not converge in 50 iterations")

predictions = np.empty(len(test))
for domain in DOMAINS:
    rows = np.flatnonzero(train_domains == domain)
    query = np.flatnonzero(test_domains == domain)
    if len(query) == 0:
        continue
    theta = fit_formula(np.stack([train_inputs[i] for i in rows]), targets[rows], domain)
    fitted = formula_inputs(np.stack([test_inputs[i] for i in query]), domain) @ theta
    predictions[query] = np.exp(fitted) if domain == "Hospital" else fitted
